In [1]:
import torch
import torch.nn as nn
import torchvision
from torchvision import transforms
from torchvision.datasets import STL10
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objs as go
import torchvision.transforms.functional as TF
from IPython.display import clear_output


In [5]:
class ResNetClassifier(nn.Module):
    def __init__(self, num_classes=10):
        super(ResNetClassifier, self).__init__()

        self.backbone = torchvision.models.resnet18(weights='DEFAULT')

        # [1] conv1 stride=1로 변경 (기존 2 → 1)
        self.backbone.conv1 = nn.Conv2d(
            in_channels=3,
            out_channels=64,
            kernel_size=7,
            stride=1,
            padding=3,
            bias=False
        )

        # [2] maxpool 제거
        self.backbone.maxpool = nn.Identity()

        # [3] fc 레이어 조정
        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Linear(in_features, num_classes)

    def forward(self, x):
        return self.backbone(x)


In [6]:
transform = transforms.Compose([
    transforms.Resize((96, 96)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

test_dataset = STL10(root='./data', split='test', download=True, transform=transform)

100%|██████████| 2.64G/2.64G [1:05:06<00:00, 676kB/s]  


In [7]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = ResNetClassifier().to(device)

model_path = r"C:\Users\orgin\XAI-study\heatmap_tool\checkpoints\resnet18\stl10.pt"
model.load_state_dict(torch.load(model_path, map_location=device))
model.eval()


ResNetClassifier(
  (backbone): ResNet(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): Identity()
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
  

In [8]:
class_names = ['airplane', 'bird', 'car', 'cat', 'deer', 'dog', 'horse', 'monkey', 'ship', 'truck']
basis_class = 'dog'
target_class = 'cat'
basis_idx = class_names.index(basis_class)
target_idx = class_names.index(target_class)

logits_basis, logits_target, predictions, colors, idx_list = [], [], [], [], []

for i in range(len(test_dataset)):
    img, label = test_dataset[i]
    if label != basis_idx:
        continue

    input_tensor = img.unsqueeze(0).to(device)
    with torch.no_grad():
        output = model(input_tensor)
        logit = output.squeeze().cpu().numpy()
        pred = logit.argmax()

    logits_basis.append(logit[basis_idx])
    logits_target.append(logit[target_idx])
    predictions.append(pred)
    idx_list.append(i)

    if pred == basis_idx:
        colors.append('royalblue')
    elif pred == target_idx:
        colors.append('orangered')
    else:
        colors.append('gray')

# 이미지 저장
images = [test_dataset[i][0] for i in idx_list]


In [9]:
def show_image(idx):
    clear_output(wait=True)
    img = TF.to_pil_image(images[idx])
    plt.figure(figsize=(2, 2))
    plt.imshow(img)
    plt.axis('off')
    plt.title(f"Pred: {class_names[predictions[idx]]}", fontsize=12)
    plt.show()

def click_callback(trace, points, selector):
    for i in points.point_inds:
        show_image(i)

# Scatter plot 생성
fig = go.Figure(data=go.Scatter(
    x=logits_basis,
    y=logits_target,
    mode='markers',
    marker=dict(size=8, color=colors),
    customdata=np.arange(len(idx_list))[:, None],
    hovertemplate="Dog logit: %{x}<br>Cat logit: %{y}<extra></extra>"
))

# y = x 대각선
min_val = min(min(logits_basis), min(logits_target))
max_val = max(max(logits_basis), max(logits_target))
fig.add_trace(go.Scatter(
    x=[min_val, max_val],
    y=[min_val, max_val],
    mode='lines',
    line=dict(dash='dash', color='black'),
    showlegend=False
))

fig.data[0].on_click(click_callback)

fig.update_layout(
    title=f"Interactive Logit Plot: {basis_class} vs {target_class}",
    xaxis_title=f"{basis_class} logit",
    yaxis_title=f"{target_class} logit",
    width=700,
    height=700
)

fig.show()
